# Deep Neural Networks - CNN Assignment
## Convolutional Neural Networks: Custom CNN vs Transfer Learning


---
## Cell 1 — Install & Import All Libraries
We install `tensorflow-datasets` (tfds) which lets us download **Cats vs Dogs** directly without uploading any files.

In [ ]:

# Installing tensorflow-datasets

# ─────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "tensorflow-datasets", "--quiet"])

# ─── Libraries used ───────────────────────────────────
import numpy as np          # numerical arrays
import pandas as pd         # dataframes for comparison tables
import matplotlib.pyplot as plt  # plotting
import seaborn as sns       # prettier confusion matrix heatmaps
import time                 # to measure training duration
import json                 # to print final JSON results
import os                   # file system utilities

# ─── Sklearn Library ──────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ─── TensorFlow / Keras Library ───────────────────────────────────
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,              # Convolution layer
    MaxPooling2D,        # Max-pooling layer
    GlobalAveragePooling2D,  # GAP — MANDATORY per assignment
    Dense,               # Fully connected output layer
    BatchNormalization,  # Normalizes activations → faster convergence
    Dropout,             # Randomly drops neurons → reduces overfitting
    Input                # Explicit input specification
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical


# Setting a seed makes results reproducible across runs
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print("All libraries imported successfully!")

# ─── Check if GPU is available  ────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU available: {len(gpus) > 0}")
if gpus:
    print(f"  → {gpus[0].name}")


# --------------- Freeing up RAM--------------------
#"""import gc
#g#c.collect()
#tf.keras.backend.clear_session()"""

---
## PART 1 — Dataset Loading & Exploration
**Dataset chosen:** `cats_vs_dogs` from TensorFlow Datasets
- 2 classes: Cat (0) and Dog (1)
- ~23,000+ labelled images
- Directly available via `tfds.load()`
- Satisfies assignment requirement: ≥500 images per class

In [ ]:
# ─────────────────────────────────────────────────────────
# CONFIGURATION CONSTANTS

# ─────────────────────────────────────────────────────────
IMG_SIZE    = 160       # setting image to 160*160 pixels

BATCH_SIZE  = 32         # number of images processed at once during training
NUM_CLASSES = 2          # Cats=0, Dogs=1
CLASS_NAMES = ['Cat', 'Dog']

# ─── Custom CNN hyperparameters (per assignment spec) ─────
CNN_EPOCHS  = 12         # 12 epochs (trial and error method)
CNN_LR      = 0.0005     # learning rate = 0.0005 as tested after multiple trial and hit

# ─── Transfer Learning hyperparameters ────────────────────
TL_EPOCHS   = 15         # fewer epochs needed — as model starts pretrained
TL_LR       = 0.0001     # used smaller LR for fine-tuning

print("Configuration:")
print(f"  Image size    : {IMG_SIZE}×{IMG_SIZE}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Num classes   : {NUM_CLASSES}")
print(f"  CNN epochs    : {CNN_EPOCHS}")
print(f"  CNN LR        : {CNN_LR}")
print(f"  TL epochs     : {TL_EPOCHS}")
print(f"  TL LR         : {TL_LR}")

In [ ]:
# ─────────────────────────────────────────────────────────
# LOAD CATS vs DOGS from TensorFlow Datasets
# ─────────────────────────────────────────────────────────
print("Downloading Cats vs Dogs dataset...")
print("(First run: this may take a few minutes)")

(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    name='cats_vs_dogs',
    split=['train[:10800]', 'train[10800:12000]'],  # Custom split: 10800 train, 1200 test (90-10% split)
    as_supervised=True,
    with_info=True
)

print("Dataset loaded successfully!")
print(f"Training examples : {10800} (approx)")
print(f"Test examples     : {1200} (approx)")

In [ ]:
import tensorflow_datasets as tfds

print(f"tensorflow_datasets version: {tfds.__version__}")

In [ ]:
# ─────────────────────────────────────────────────────────
# PREPROCESSING FUNCTION
#  I applied to every image in the dataset pipeline.
# ─────────────────────────────────────────────────────────
def preprocess(image, label):
    """
    Resize image to IMG_SIZE×IMG_SIZE and normalize pixel values to [0, 1].

    Why did i normalize?
      - Raw pixels range 0–255. Neural networks train better with small values.
      - Dividing by 255.0 maps pixels to [0.0, 1.0].
    """
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])  # resizing
    image = tf.cast(image, tf.float32) / 255.0            # normalizing
    return image, label

# ─── Applying preprocessing and creating NumPy arrays ──────────
# I converted from tf.data pipeline to NumPy arrays so we can
# use sklearn metrics (accuracy_score etc.) easily.

def dataset_to_numpy(ds):
    """
    Converting a tf.data.Dataset of (image, label) to two NumPy arrays.
    I collect all batches and stack them.
    """
    images, labels = [], []
    for img, lbl in ds.map(preprocess).batch(64):
        images.append(img.numpy())
        labels.append(lbl.numpy())
    return np.concatenate(images), np.concatenate(labels)

print("Preprocessing training data...")
X_train, y_train_int = dataset_to_numpy(ds_train_raw)

print("Preprocessing test data...")
X_test, y_test_int  = dataset_to_numpy(ds_test_raw)

print(f"X_train shape : {X_train.shape}  | dtype: {X_train.dtype}")
print(f"X_test  shape : {X_test.shape}")
print(f"Pixel range   : [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# ─────────────────────────────────────────────────────────
# ONE-HOT ENCODE LABELS
# Neural network output layer has 2 neurons (softmax).
# We need labels as [1,0] for Cat and [0,1] for Dog.
# to_categorical does this automatically.
# ─────────────────────────────────────────────────────────
y_train_cat = to_categorical(y_train_int, num_classes=NUM_CLASSES)  # shape: (N, 2)
y_test_cat  = to_categorical(y_test_int,  num_classes=NUM_CLASSES)

print(f"y_train_cat shape : {y_train_cat.shape}   (one-hot encoded)")
print(f"y_test_cat  shape : {y_test_cat.shape}")
print(f"Example label     : int={y_train_int[0]}, one-hot={y_train_cat[0]}")

In [ ]:
# ─────────────────────────────────────────────────────────
# METADATA VARIABLES
# ─────────────────────────────────────────────────────────
dataset_name    = "cats_vs_dogs"
dataset_source  = "tensorflow_datasets (tfds)"
n_samples       = len(X_train) + len(X_test)
n_classes       = NUM_CLASSES
train_samples   = len(X_train)
test_samples    = len(X_test)
train_test_ratio = "90/10"
image_shape     = [IMG_SIZE, IMG_SIZE, 3]
problem_type    = "classification"

# Counting samples per class in training set
unique, counts = np.unique(y_train_int, return_counts=True)
spc_min = counts.min()
spc_max = counts.max()
spc_avg = counts.mean()
samples_per_class = f"min:{spc_min}, max:{spc_max}, avg:{spc_avg:.0f}"

# Primary metric choice & justification
primary_metric = "accuracy"
metric_justification = (
    "Cats vs Dogs is a balanced binary classification dataset with roughly equal "
    "samples per class, so accuracy is a fair and intuitive primary metric. "
    "Neither false positives nor false negatives carry disproportionate real-world cost here."
)

print("=" * 70)
print("DATASET INFORMATION")
print(f"Dataset          : {dataset_name}")
print(f"Source           : {dataset_source}")
print(f"Total Samples    : {n_samples}")
print(f"Number of Classes: {n_classes}")
print(f"Samples/Class    : {samples_per_class}")
print(f"Image Shape      : {image_shape}")
print(f"Train/Test Split : {train_test_ratio}")
print(f"Train Samples    : {train_samples}")
print(f"Test Samples     : {test_samples}")
print(f"Primary Metric   : {primary_metric}")
print(f"Justification    : {metric_justification}")
print("=" * 70)

In [ ]:
# ─────────────────────────────────────────────────────────
# EXPLORATORY DATA ANALYSIS (EDA)
# Showing sample images and plotting class distribution.
# ─────────────────────────────────────────────────────────

# 1. Sample images
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Sample Images from Cats vs Dogs Dataset", fontsize=14, fontweight='bold')

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    # Find indices belonging to this class
    cls_indices = np.where(y_train_int == cls_idx)[0][:5]
    for col, idx in enumerate(cls_indices):
        axes[cls_idx, col].imshow(X_train[idx])
        axes[cls_idx, col].set_title(cls_name, fontsize=10)
        axes[cls_idx, col].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print("Sample images displayed.")

# 2. Class distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training set distribution
train_counts = [np.sum(y_train_int == i) for i in range(NUM_CLASSES)]
axes[0].bar(CLASS_NAMES, train_counts, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Training Set Class Distribution')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate(train_counts):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Test set distribution
test_counts = [np.sum(y_test_int == i) for i in range(NUM_CLASSES)]
axes[1].bar(CLASS_NAMES, test_counts, color=['#4C72B0', '#DD8452'])
axes[1].set_title('Test Set Class Distribution')
axes[1].set_ylabel('Number of Images')
for i, v in enumerate(test_counts):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("Class distribution plotted.")
print(f"Training  → Cats: {train_counts[0]}, Dogs: {train_counts[1]}")
print(f"Test      → Cats: {test_counts[0]}, Dogs: {test_counts[1]}")

---
## PART 2 — Custom CNN Implementation

**Architecture:** 4 Conv2D blocks → GlobalAveragePooling2D → Dense(softmax)

| Layer Type        | Filters | Kernel | Output Shape | Purpose                      |
|-------------------|---------|--------|--------------|------------------------------|
| Input             | —       | —      | 160×160×3    | RGB image input              |
| RandomFlip        | —       | —      | 160×160×3    | Horizontal flip augmentation |
| RandomRotation    | —       | —      | 160×160×3    | ±10% rotation augmentation   |
| RandomZoom        | —       | —      | 160×160×3    | ±10% zoom augmentation       |
| Conv2D Block 1    | 32      | 3×3    | 80×80×32     | Edge detection               |
| Conv2D Block 2    | 64      | 3×3    | 40×40×64     | Texture features             |
| Conv2D Block 3    | 128     | 3×3    | 20×20×128    | Complex shapes               |
| Conv2D Block 4    | 256     | 3×3    | 10×10×256    | High-level semantics         |
| GlobalAvgPool2D   | —       | —      | 256          | Replaces Flatten (MANDATORY) |
| Dropout(0.4)      | —       | —      | 256          | Regularization               |
| Dense (softmax)   | 2       | —      | 2            | Class probabilities          |2               | Class probabilities |

In [ ]:
def build_custom_cnn(input_shape, n_classes):
    """
    I built a custom 4-block CNN.

    Architecture:
      Augmentation → [Conv2D → BatchNorm → MaxPool] × 4 → GAP → Dropout → Dense(softmax)

    Args:
        input_shape : tuple, e.g. (160, 160, 3)
        n_classes   : int, number of output classes
    Returns:
        model       : compiled Keras model
    """
    model = Sequential(name="Custom_CNN")

    # ── INPUT LAYER ────────────────────────────────────────
    model.add(Input(shape=input_shape))

    # ── DATA AUGMENTATION ──────────────────────────────────
    # These layers are ONLY active during training.
    # Automatically turned OFF during model.predict() and model.evaluate().
    # Artificially increases dataset variety → reduces overfitting.
    model.add(tf.keras.layers.RandomFlip("horizontal"))  # randomly mirror image left-right
    model.add(tf.keras.layers.RandomRotation(0.1))       # randomly rotate up to ±10%
    model.add(tf.keras.layers.RandomZoom(0.1))           # randomly zoom in/out up to 10%

    # ── BLOCK 1: 32 filters ────────────────────────────────
    # Conv2D: 32 filters of size 3×3, 'same' padding keeps output same size as input.
    # ReLU activation: max(0, x) — removes negatives, avoids vanishing gradients.
    model.add(Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu',
                     name='conv1'))
    # BatchNormalization: normalizes activations within each mini-batch.
    # Speeds up training and acts as mild regularizer.
    model.add(BatchNormalization(name='bn1'))
    # MaxPooling2D: takes the maximum value in each 2×2 window.
    # Halves spatial dimensions (160→80) while keeping dominant features.
    model.add(MaxPooling2D(pool_size=(2,2), name='pool1'))

    # ── BLOCK 2: 64 filters ────────────────────────────────
    model.add(Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu',
                     name='conv2'))
    model.add(BatchNormalization(name='bn2'))
    model.add(MaxPooling2D(pool_size=(2,2), name='pool2'))  # 80→40

    # ── BLOCK 3: 128 filters ───────────────────────────────
    model.add(Conv2D(filters=128, kernel_size=(3,3), padding='same', activation='relu',
                     name='conv3'))
    model.add(BatchNormalization(name='bn3'))
    model.add(MaxPooling2D(pool_size=(2,2), name='pool3'))  # 40→20

    # ── BLOCK 4: 256 filters ───────────────────────────────
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding='same', activation='relu',
                     name='conv4'))
    model.add(BatchNormalization(name='bn4'))
    model.add(MaxPooling2D(pool_size=(2,2), name='pool4'))  # 20→10

    # ── GLOBAL AVERAGE POOLING (GAP) — MANDATORY ───────────
    # Instead of Flatten (which would give 10×10×256 = 25,600 values),
    # GAP computes the mean of each feature map → gives 256 values.
    # This dramatically reduces parameters and prevents overfitting.
    model.add(GlobalAveragePooling2D(name='gap'))           # output: (256,)

    # ── DROPOUT ────────────────────────────────────────────
    # Randomly sets 40% of neurons to 0 during training.
    # Forces the network not to rely on any single neuron → reduces overfitting.
    model.add(Dropout(rate=0.4, name='dropout'))

    # ── OUTPUT LAYER ───────────────────────────────────────
    # Dense with 'softmax' converts raw scores into probabilities.
    # For 2 classes: outputs [P(Cat), P(Dog)] summing to 1.0.
    model.add(Dense(units=n_classes, activation='softmax', name='output'))

    # ── COMPILE ────────────────────────────────────────────
    # Adam optimizer with LR 0.0005
    # categorical_crossentropy is standard for multi-class one-hot targets
    model.compile(
        optimizer=Adam(learning_rate=CNN_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ─── Creating the model ───────────────────────────────────
input_shape = (IMG_SIZE, IMG_SIZE, 3)   # (160, 160, 3)
custom_cnn  = build_custom_cnn(input_shape, NUM_CLASSES)

# Printing summary — shows each layer, output shape, and parameter count
custom_cnn.summary()

In [ ]:
# ─────────────────────────────────────────────────────────
# CALLBACKS
# Callbacks are actions that Keras performs at the end of each epoch.
# ─────────────────────────────────────────────────────────

# EarlyStopping: if validation loss doesn't improve for 5 epochs (to save time and fine-tune), stop.
# restore_best_weights=True: reverts to the best model weights automatically.
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

# ReduceLROnPlateau: if val_loss plateaus for 3 epochs, halve the learning rate.
# This helps the model escape local minima.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("Callbacks created: EarlyStopping (patience=20), ReduceLROnPlateau (patience=3)")

In [ ]:
# ─────────────────────────────────────────────────────────
# TRAINING CUSTOM CNN
# model.fit() runs the training loop:
#   1. Forward pass: compute predictions
#   2. Compute loss (how wrong the predictions are)
#   3. Backward pass: compute gradients via backpropagation
#   4. Update weights using Adam optimizer
#   Repeating for each batch, then move to next epoch.
# ─────────────────────────────────────────────────────────
print("=" * 70)
print("CUSTOM CNN TRAINING")
print(f"Epochs: {CNN_EPOCHS} | LR: {CNN_LR} | Batch: {BATCH_SIZE}")
print("=" * 70)

custom_cnn_start_time = time.time()

cnn_history = custom_cnn.fit(
    X_train, y_train_cat,          # passing training data
    epochs=CNN_EPOCHS,             # 30 epochs
    batch_size=BATCH_SIZE,         # 32 images per step
    validation_split=0.1,          # use 10% of training data for validation
    callbacks=[early_stop, reduce_lr],
    verbose=1                      # print progress per epoch
)

custom_cnn_training_time = time.time() - custom_cnn_start_time

# ─── Extract initial and final training loss ──────────────
# cnn_history.history['loss'] is a list of loss values, one per epoch.
custom_cnn_initial_loss = float(cnn_history.history['loss'][0])
custom_cnn_final_loss   = float(cnn_history.history['loss'][-1])

# Convergence check
reduction_pct = (custom_cnn_initial_loss - custom_cnn_final_loss) / custom_cnn_initial_loss * 100

print(f"\nTraining completed in {custom_cnn_training_time:.2f} seconds")
print(f"Initial Loss  : {custom_cnn_initial_loss:.4f}")
print(f"Final Loss    : {custom_cnn_final_loss:.4f}")
print(f"Loss Reduction: {reduction_pct:.1f}%", end=" ")
if reduction_pct >= 50:
    print("Loss reduction is greater than or equal to 50%")
elif reduction_pct >= 20:
    print("Loss reduction lies between 50% and 20%")
else:
    print("Loss reduction is below 20%")

In [ ]:
# ─────────────────────────────────────────────────────────
# EVALUATING CUSTOM CNN ON TEST SET
# ─────────────────────────────────────────────────────────
print("=" * 70)
print("CUSTOM CNN EVALUATION")
print("=" * 70)

# model.predict() returns a probability array of shape (N, 2)

cnn_pred_probs = custom_cnn.predict(X_test, verbose=0)

# np.argmax: take the class with the highest probability as prediction
# e.g. [0.8, 0.2] → argmax = 0 (Cat)
cnn_pred_labels = np.argmax(cnn_pred_probs, axis=1)
# y_test_int already contains true integer labels [0, 1, 0, ...]

# ── Calculating all 4 required metrics ──────────────────────
# All values are in [0, 1] range as required.
# average='macro' = unweighted mean across classes (fair for balanced data)
custom_cnn_accuracy  = float(accuracy_score(y_test_int, cnn_pred_labels))
custom_cnn_precision = float(precision_score(y_test_int, cnn_pred_labels, average='macro'))
custom_cnn_recall    = float(recall_score(y_test_int, cnn_pred_labels, average='macro'))
custom_cnn_f1        = float(f1_score(y_test_int, cnn_pred_labels, average='macro'))

print("\nCustom CNN Performance on Test Set:")
print(f"  Accuracy  : {custom_cnn_accuracy:.4f}")
print(f"  Precision : {custom_cnn_precision:.4f}")
print(f"  Recall    : {custom_cnn_recall:.4f}")
print(f"  F1-Score  : {custom_cnn_f1:.4f}")

# ── Detailed per-class report ──────────────────────────────
print("\nClassification Report:")
print(classification_report(y_test_int, cnn_pred_labels, target_names=CLASS_NAMES))

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZING CUSTOM CNN RESULTS
# 1. Plotting Training curves (loss + accuracy)
# 2. Displaying Confusion matrix
# ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Custom CNN — Training Results", fontsize=14, fontweight='bold')

epochs_ran = range(1, len(cnn_history.history['loss']) + 1)

# ── Plot 1: Training & Validation Loss ────────────────────
axes[0].plot(epochs_ran, cnn_history.history['loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_ran, cnn_history.history['val_loss'], 'r-o', label='Val Loss', markersize=4)
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Training & Validation Accuracy ────────────────
axes[1].plot(epochs_ran, cnn_history.history['accuracy'], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(epochs_ran, cnn_history.history['val_accuracy'], 'r-o', label='Val Acc', markersize=4)
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── Plot 3: Confusion Matrix ───────────────────────────────
cm = confusion_matrix(y_test_int, cnn_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[2])
axes[2].set_title('Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('custom_cnn_results.png', dpi=100, bbox_inches='tight')
plt.show()
print("Custom CNN visualizations saved.")

---
## PART 3 — Transfer Learning with ResNet50

**Strategy:**
1. Loading ResNet50 pretrained on ImageNet (1.2M images, 1000 classes)
2. **Freezing all base layers** — they already know edges, textures, shapes
3. Adding **GlobalAveragePooling2D** → Dense(softmax)
4. Train only the new head on Cats vs Dogs

In [ ]:
# ─────────────────────────────────────────────────────────
# BUILDING TRANSFER LEARNING MODEL
# ─────────────────────────────────────────────────────────
print("=" * 70)
print("TRANSFERING LEARNING IMPLEMENTATION")
print("=" * 70)

pretrained_model_name = "ResNet50"

def build_transfer_learning_model(input_shape, n_classes):
    """
    Building transfer learning model using ResNet50 as feature extractor.

    Steps:
      1. Loading ResNet50 WITHOUT its top classification layers (include_top=False)
      2. Freezing ALL base layers
      3. Adding GAP + custom Dense head

    Args:
        input_shape : tuple, e.g. (128, 128, 3)
        n_classes   : int
    Returns:
        model       : compiled Keras functional model
    """
    # ── LOADING PRETRAINED BASE ───────────────────────────────
    # weights='imagenet' : download ImageNet weights (~100 MB, done once)
    # include_top=False  : remove the original 1000-class Dense head
    # input_shape        : resize inputs to match our 128×128
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # ── FREEZING ALL BASE LAYERS ─────────────────────────────
    # trainable=False means: during training, don't update these weights.
    # The pretrained features are preserved exactly as ImageNet trained them.
    base_model.trainable = False

    # ── BUILDING CUSTOM HEAD using Functional API ─────────────
    # Functional API allows non-sequential connections (vs Sequential above)
    inputs = Input(shape=input_shape)

    # Passing inputs through frozen ResNet50 — it acts as a feature extractor
    # training=False ensures BatchNorm layers in ResNet use stored statistics
    x = base_model(inputs, training=False)

    # GlobalAveragePooling2D
    # ResNet50 output: (4, 4, 2048) — GAP converts to (2048,)
    x = GlobalAveragePooling2D(name='transfer_gap')(x)

    # Dropout for regularization
    x = Dropout(0.3, name='transfer_dropout')(x)

    # Final classification layer
    outputs = Dense(n_classes, activation='softmax', name='transfer_output')(x)

    model = Model(inputs, outputs, name="ResNet50_Transfer")

    # Compiling with lower LR (fine-tuning requires careful, small updates)
    model.compile(
        optimizer=Adam(learning_rate=TL_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, base_model


transfer_model, base_resnet = build_transfer_learning_model(input_shape, NUM_CLASSES)
transfer_model.summary()

In [ ]:
# ─────────────────────────────────────────────────────────
# COUNTING LAYER STATISTICS
# ─────────────────────────────────────────────────────────

# Count frozen vs trainable layers in the base model
frozen_layers    = sum(1 for l in base_resnet.layers if not l.trainable)
trainable_layers = sum(1 for l in transfer_model.layers if l.trainable)

# Count parameters in the full model
total_parameters     = transfer_model.count_params()
trainable_parameters = sum(tf.keras.backend.count_params(w)
                           for w in transfer_model.trainable_weights)

print(f"Base Model         : {pretrained_model_name}")
print(f"Frozen Layers      : {frozen_layers}")
print(f"Trainable Layers   : {trainable_layers}")
print(f"Total Parameters   : {total_parameters:,}")
print(f"Trainable Params   : {trainable_parameters:,}")
print(f"Using GAP          : YES  ")

In [ ]:
# ─────────────────────────────────────────────────────────
# TRAINING TRANSFER LEARNING MODEL
# ─────────────────────────────────────────────────────────
print("=" * 70)
print("TRANSFER LEARNING TRAINING")
print(f"Epochs: {TL_EPOCHS} | LR: {TL_LR} | Batch: {BATCH_SIZE}")
print("=" * 70)

tl_learning_rate = TL_LR
tl_epochs        = TL_EPOCHS
tl_batch_size    = BATCH_SIZE
tl_optimizer     = "Adam"

# Separate callbacks for transfer learning model
tl_early_stop = EarlyStopping(monitor='val_loss', patience=4,
                               restore_best_weights=True, verbose=1)
tl_reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                   patience=2, min_lr=1e-7, verbose=1)

tl_start_time = time.time()

tl_history = transfer_model.fit(
    X_train, y_train_cat,
    epochs=tl_epochs,
    batch_size=tl_batch_size,
    validation_split=0.1,
    callbacks=[tl_early_stop, tl_reduce_lr],
    verbose=1
)

tl_training_time = time.time() - tl_start_time

tl_initial_loss = float(tl_history.history['loss'][0])
tl_final_loss   = float(tl_history.history['loss'][-1])

tl_reduction_pct = (tl_initial_loss - tl_final_loss) / tl_initial_loss * 100

print(f"\nTraining completed in {tl_training_time:.2f} seconds")
print(f"Initial Loss  : {tl_initial_loss:.4f}")
print(f"Final Loss    : {tl_final_loss:.4f}")
print(f"Loss Reduction: {tl_reduction_pct:.1f}%")

In [ ]:
# ─────────────────────────────────────────────────────────
# EVALUATING TRANSFER LEARNING MODEL
# ─────────────────────────────────────────────────────────
print("=" * 70)
print("TRANSFER LEARNING EVALUATION")
print("=" * 70)

tl_pred_probs  = transfer_model.predict(X_test, verbose=0)
tl_pred_labels = np.argmax(tl_pred_probs, axis=1)

tl_accuracy  = float(accuracy_score(y_test_int, tl_pred_labels))
tl_precision = float(precision_score(y_test_int, tl_pred_labels, average='macro'))
tl_recall    = float(recall_score(y_test_int, tl_pred_labels, average='macro'))
tl_f1        = float(f1_score(y_test_int, tl_pred_labels, average='macro'))

print("\nTransfer Learning Performance on Test Set:")
print(f"  Accuracy  : {tl_accuracy:.4f}")
print(f"  Precision : {tl_precision:.4f}")
print(f"  Recall    : {tl_recall:.4f}")
print(f"  F1-Score  : {tl_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_int, tl_pred_labels, target_names=CLASS_NAMES))

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZING TRANSFER LEARNING RESULTS
# ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ResNet50 Transfer Learning — Training Results", fontsize=14, fontweight='bold')

tl_epochs_ran = range(1, len(tl_history.history['loss']) + 1)

axes[0].plot(tl_epochs_ran, tl_history.history['loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(tl_epochs_ran, tl_history.history['val_loss'], 'r-o', label='Val Loss', markersize=4)
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(tl_epochs_ran, tl_history.history['accuracy'], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(tl_epochs_ran, tl_history.history['val_accuracy'], 'r-o', label='Val Acc', markersize=4)
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

tl_cm = confusion_matrix(y_test_int, tl_pred_labels)
sns.heatmap(tl_cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[2])
axes[2].set_title('Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('transfer_learning_results.png', dpi=100, bbox_inches='tight')
plt.show()
print("Transfer learning visualizations saved.")

---
## PART 4 — Model Comparison & Visualization

In [ ]:
# ─────────────────────────────────────────────────────────
# COMPARISON TABLE
# ─────────────────────────────────────────────────────────
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

cnn_total_params = custom_cnn.count_params()

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score',
               'Training Time (s)', 'Total Parameters', 'Initial Loss', 'Final Loss'],
    'Custom CNN (4-block)': [
        f"{custom_cnn_accuracy:.4f}",
        f"{custom_cnn_precision:.4f}",
        f"{custom_cnn_recall:.4f}",
        f"{custom_cnn_f1:.4f}",
        f"{custom_cnn_training_time:.1f}",
        f"{cnn_total_params:,}",
        f"{custom_cnn_initial_loss:.4f}",
        f"{custom_cnn_final_loss:.4f}"
    ],
    'ResNet50 Transfer': [
        f"{tl_accuracy:.4f}",
        f"{tl_precision:.4f}",
        f"{tl_recall:.4f}",
        f"{tl_f1:.4f}",
        f"{tl_training_time:.1f}",
        f"{total_parameters:,}",
        f"{tl_initial_loss:.4f}",
        f"{tl_final_loss:.4f}"
    ]
})

print(comparison_df.to_string(index=False))

# ── Side-by-side bar chart of 4 metrics ───────────────────
metrics      = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
cnn_scores   = [custom_cnn_accuracy, custom_cnn_precision, custom_cnn_recall, custom_cnn_f1]
tl_scores    = [tl_accuracy, tl_precision, tl_recall, tl_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, cnn_scores, width, label='Custom CNN',       color='#4C72B0')
bars2 = ax.bar(x + width/2, tl_scores,  width, label='ResNet50 Transfer', color='#55A868')

ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Adding value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Comparison chart saved.")

In [ ]:
# ─────────────────────────────────────────────────────────
# SIDE-BY-SIDE TRAINING CURVES
# ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Loss Comparison", fontsize=13, fontweight='bold')

# Custom CNN
axes[0].plot(range(1, len(cnn_history.history['loss'])+1),
             cnn_history.history['loss'], 'b-', label='Train Loss')
axes[0].plot(range(1, len(cnn_history.history['val_loss'])+1),
             cnn_history.history['val_loss'], 'b--', label='Val Loss')
axes[0].set_title('Custom CNN Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Transfer Learning
axes[1].plot(range(1, len(tl_history.history['loss'])+1),
             tl_history.history['loss'], 'g-', label='Train Loss')
axes[1].plot(range(1, len(tl_history.history['val_loss'])+1),
             tl_history.history['val_loss'], 'g--', label='Val Loss')
axes[1].set_title('ResNet50 Transfer Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# SAMPLE PREDICTIONS WITH IMAGES
# ─────────────────────────────────────────────────────────
def plot_predictions(model, X, y_true, class_names, n=20, title='Predictions'):
    """
    Showing n test images with their true and predicted labels.
    Correct predictions: green border. Wrong: red.
    """
    preds = np.argmax(model.predict(X[:n], verbose=0), axis=1)
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for i, ax in enumerate(axes.flat):
        ax.imshow(X[i])
        color = 'green' if preds[i] == y_true[i] else 'red'
        ax.set_title(f"T:{class_names[y_true[i]]}\nP:{class_names[preds[i]]}",
                     color=color, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'predictions_{title.replace(" ","_").lower()}.png',
                dpi=80, bbox_inches='tight')
    plt.show()

plot_predictions(custom_cnn,    X_test, y_test_int, CLASS_NAMES, title='Custom CNN Predictions')
plot_predictions(transfer_model, X_test, y_test_int, CLASS_NAMES, title='Transfer Learning Predictions')

---
## PART 5 — Analysis

In [ ]:
# ─────────────────────────────────────────────────────────
# ANALYSIS TEXT
# Covers all 6 required topics for full marks.
# ─────────────────────────────────────────────────────────
analysis_text = """
The custom 4-block CNN outperformed the ResNet50 transfer learning model across all
metrics. Custom CNN achieved better accuracy, precision, recall,
and F1-score, versus transfer learning — a gap of approximately 23 percentage points.

The primary reason for transfer learning's underperformance is the image resolution
constraint. ResNet50 was pretrained on 160*160 ImageNet images; however, due to
computational limitations on Google Colab T4 GPU (12.7 GB RAM), input size was
restricted to 160×160 pixels — as sizes 198 and above caused session crashes,
with 160×160 itself consuming 10.7 GB RAM. This resolution mismatch distorted
pretrained filter activations in deeper ResNet layers, limiting feature extraction
quality and resulting in only 8.5% loss reduction versus the custom CNN's 57.7%.

Global Average Pooling proved critical in both models. Despite the custom CNN having
only 390,850 parameters — 60× fewer than ResNet50's 23.5M — GAP prevented
overfitting by averaging spatial feature maps instead of flattening them, enabling
strong generalization from limited spatial resolution.

Training times were comparable (~304s vs ~298s), demonstrating that frozen base
layers in ResNet50 eliminate backpropagation overhead, making parameter count
irrelevant to training speed. Transfer learning excels when input resolution matches
the pretrained domain; custom CNNs are more robust under hardware constraints.
"""

print("=" * 60)
print("ANALYSIS")
print(analysis_text)
word_count = len(analysis_text.split())
print(f"Analysis word count: {word_count} words")
if word_count > 200:
    print("⚠  Warning: Analysis exceeds 200 words (no marks deducted)")
else:
    print("✔  Analysis within word count guideline")

---
## PART 6 — Code Structure Check & JSON Results


In [ ]:
# ─────────────────────────────────────────────────────────
# IMPLEMENTATION VERIFICATION
# Confirm both models use GAP and not Flatten.
# ─────────────────────────────────────────────────────────
def check_gap_usage(model, model_name):
    layer_types = [type(l).__name__ for l in model.layers]
    has_gap     = 'GlobalAveragePooling2D' in layer_types
    has_flatten = 'Flatten' in layer_types
    print(f"{model_name}:")
    print(f"  GlobalAveragePooling2D present : {'YES ✔' if has_gap     else 'NO ✗'}")
    print(f"  Flatten present (should be NO): {'YES ✗' if has_flatten  else 'NO ✔'}")
    return has_gap and not has_flatten

print("=" * 60)
print("IMPLEMENTATION VERIFICATION")
cnn_ok = check_gap_usage(custom_cnn,    "Custom CNN")
tl_ok  = check_gap_usage(transfer_model, "ResNet50 Transfer")
print(f"\nBoth models valid: {'YES ✔' if cnn_ok and tl_ok else 'NO ✗'}")

In [ ]:
# ─────────────────────────────────────────────────────────
# ASSIGNMENT RESULTS JSON  — DO NOT MODIFY STRUCTURE
# ─────────────────────────────────────────────────────────
def get_assignment_results():
    """
    Generate complete assignment results in required format.
    DO NOT MODIFY field names — autograder relies on exact keys.
    """
    framework_used = "keras"

    results = {
        # ── Dataset Information ────────────────────────────
        'dataset_name'       : dataset_name,
        'dataset_source'     : dataset_source,
        'n_samples'          : n_samples,
        'n_classes'          : n_classes,
        'samples_per_class'  : samples_per_class,
        'image_shape'        : image_shape,
        'problem_type'       : problem_type,
        'primary_metric'     : primary_metric,
        'metric_justification': metric_justification,
        'train_samples'      : train_samples,
        'test_samples'       : test_samples,
        'train_test_ratio'   : train_test_ratio,

        # ── Custom CNN Results ─────────────────────────────
        'custom_cnn': {
            'framework'  : framework_used,
            'architecture': {
                'conv_layers'               : 4,
                'pooling_layers'            : 4,
                'has_global_average_pooling': True,
                'output_layer'              : 'softmax',
                'total_parameters'          : int(custom_cnn.count_params())
            },
            'training_config': {
                'learning_rate' : CNN_LR,
                'n_epochs'      : CNN_EPOCHS,
                'batch_size'    : BATCH_SIZE,
                'optimizer'     : 'Adam',
                'loss_function' : 'categorical_crossentropy'
            },
            'initial_loss'         : custom_cnn_initial_loss,
            'final_loss'           : custom_cnn_final_loss,
            'training_time_seconds': custom_cnn_training_time,
            'accuracy'   : custom_cnn_accuracy,
            'precision'  : custom_cnn_precision,
            'recall'     : custom_cnn_recall,
            'f1_score'   : custom_cnn_f1
        },

        # ── Transfer Learning Results ──────────────────────
        'transfer_learning': {
            'framework'              : framework_used,
            'base_model'             : pretrained_model_name,
            'frozen_layers'          : frozen_layers,
            'trainable_layers'       : trainable_layers,
            'has_global_average_pooling': True,
            'total_parameters'       : int(total_parameters),
            'trainable_parameters'   : int(trainable_parameters),
            'training_config': {
                'learning_rate' : tl_learning_rate,
                'n_epochs'      : tl_epochs,
                'batch_size'    : tl_batch_size,
                'optimizer'     : tl_optimizer,
                'loss_function' : 'categorical_crossentropy'
            },
            'initial_loss'         : tl_initial_loss,
            'final_loss'           : tl_final_loss,
            'training_time_seconds': tl_training_time,
            'accuracy'   : tl_accuracy,
            'precision'  : tl_precision,
            'recall'     : tl_recall,
            'f1_score'   : tl_f1
        },

        # ── Analysis ───────────────────────────────────────
        'analysis'           : analysis_text,
        'analysis_word_count': len(analysis_text.split()),

        # ── Training Success Indicators ────────────────────
        'custom_cnn_loss_decreased'       : custom_cnn_final_loss < custom_cnn_initial_loss,
        'transfer_learning_loss_decreased': tl_final_loss < tl_initial_loss
    }

    return results


# ─── Generate and print ────────────────────────────────────
try:
    assignment_results = get_assignment_results()
    print("\n" + "=" * 70)
    print("ASSIGNMENT RESULTS SUMMARY")
    print("=" * 70)
    print(json.dumps(assignment_results, indent=2))
    print("\n✔ JSON output generated successfully — autograder ready!")
except Exception as e:
    print(f"\n❌ ERROR generating results: {str(e)}")
    print("Please ensure all variables are properly defined.")

In [ ]:
# ─────────────────────────────────────────────────────────
# ENVIRONMENT INFORMATION
# ─────────────────────────────────────────────────────────
import platform
import sys
from datetime import datetime

print("ENVIRONMENT INFORMATION")
print(f"Python version     : {sys.version.split()[0]}")
print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Platform           : {platform.platform()}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

